In [9]:
import pandas as pd
import numpy as np
import io
import os

# ============================================================================
# 0. SET YOUR PATHS HERE
# ============================================================================
# Folder containing all your raw CSVs (adjust if your files live elsewhere)
DATA_DIR = r"C:\Users\lenovo\OneDrive\Desktop\Folder\cab_bike_auto_fare_prediction\dataset"

path = DATA_DIR + os.sep

COMMON_COLS = [
    "source_dataset", "date", "time", "vehicle_type",
    "pickup_location", "drop_location",
    "distance_km", "fare_amount", "payment_method",
    "status", "driver_rating", "customer_rating"
]

def make_common(df):
    for c in COMMON_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[COMMON_COLS]

# ============================================================================
# 1. LOAD RAW FILES
# ============================================================================
print("Loading raw files from:", DATA_DIR)

df_ola      = pd.read_csv(path + "Bengaluru_ola.csv")
df_bookings = pd.read_csv(path + "Bookings.csv")
df_ncr      = pd.read_csv(path + "ncr_rides_final_event_dataset_2024.csv")
df_rides    = pd.read_csv(path + "rides_data.csv")
df_delhi    = pd.read_csv(path + "delhi_fare_rates.csv")
df_aru      = pd.read_csv(path + "arunachal_fare_rates.csv")

# --- Indore Ola needs a special loader (rows are double-quote wrapped) ---
def load_indore_ola(filepath):
    with open(filepath, "r", encoding="utf-8", newline="") as f:
        raw_lines = f.read().splitlines()
    cleaned_lines = []
    for ln in raw_lines:
        ln = ln.strip()
        if ln.startswith('"') and ln.endswith('"'):
            ln = ln[1:-1]
        cleaned_lines.append(ln)
    return pd.read_csv(io.StringIO("\n".join(cleaned_lines)))

df_indore = load_indore_ola(path + "indore_ola_dataset.csv")

# ============================================================================
# 2. CLEAN + STANDARDIZE EACH DATASET INDIVIDUALLY
# ============================================================================

# --- 2a. Bengaluru_ola.csv ---
d = df_ola.copy()
d["source_dataset"]   = "bengaluru_ola"
d["date"]             = pd.to_datetime(d["Date"], format="%d/%m/%Y", errors="coerce")
d["time"]             = d["Time"]
d["vehicle_type"]     = d["Vehicle Type"]
d["pickup_location"]  = d["Pickup Location"]
d["drop_location"]    = d["Drop Location"]
d["distance_km"]      = pd.to_numeric(d["Ride Distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["Booking Value"], errors="coerce")
d["payment_method"]   = d["Payment Method"]
d["status"]           = d["Booking Status"]
d["driver_rating"]    = pd.to_numeric(d["Driver Ratings"], errors="coerce")
d["customer_rating"]  = pd.to_numeric(d["Customer Rating"], errors="coerce")
df_ola_clean = make_common(d)

# --- 2b. Bookings.csv ---
d = df_bookings.copy()
d.columns = d.columns.str.strip()
d["source_dataset"]   = "bookings"
d["date"]             = pd.to_datetime(d["Date"], errors="coerce")
d["time"]             = d["Time"]
d["vehicle_type"]     = d["Vehicle_Type"]
d["pickup_location"]  = d["Pickup_Location"]
d["drop_location"]    = d["Drop_Location"]
d["distance_km"]      = pd.to_numeric(d["Ride_Distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["Booking_Value"], errors="coerce")
d["payment_method"]   = d["Payment_Method"]
d["status"]           = d["Booking_Status"]
d["driver_rating"]    = pd.to_numeric(d["Driver_Ratings"], errors="coerce")
d["customer_rating"]  = pd.to_numeric(d["Customer_Rating"], errors="coerce")
df_bookings_clean = make_common(d)

# --- 2c. ncr_rides_final_event_dataset_2024.csv ---
d = df_ncr.copy()
d["source_dataset"]   = "ncr_events"
d["date"]             = pd.to_datetime(d["Date"], errors="coerce")
d["time"]             = d["Time"]
d["vehicle_type"]     = d["Vehicle Type"]
d["pickup_location"]  = d["Pickup Location"]
d["drop_location"]    = d["Drop Location"]
d["distance_km"]      = pd.to_numeric(d["Ride Distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["Booking Value"], errors="coerce")
d["payment_method"]   = d["Payment Method"]
d["status"]           = d["Booking Status"]
d["driver_rating"]    = pd.to_numeric(d["Driver Ratings"], errors="coerce")
d["customer_rating"]  = pd.to_numeric(d["Customer Rating"], errors="coerce")
df_ncr_clean = make_common(d)

# --- 2d. rides_data.csv ---
d = df_rides.copy()
d["source_dataset"]   = "rides_data"
d["date"]             = pd.to_datetime(d["date"], errors="coerce")
d["time"]             = d["time"]
d["vehicle_type"]     = d["services"]
d["pickup_location"]  = d["source"]
d["drop_location"]    = d["destination"]
d["distance_km"]      = pd.to_numeric(d["distance"], errors="coerce")
d["fare_amount"]      = pd.to_numeric(d["total_fare"], errors="coerce")
d["payment_method"]   = d["payment_method"]
d["status"]           = d["ride_status"]
df_rides_clean = make_common(d)

# --- 2e. Delhi official fare-rate table ---
d = df_delhi.copy()
d["source_dataset"]  = "delhi_notification_2023"
d["date"]            = pd.NaT
d["time"]            = np.nan
d["vehicle_type"] = np.where(
    d["vehicle_type"] == "taxi",
    "taxi_" + d["ac_type"].astype(str),
    d["vehicle_type"]
)
d["pickup_location"] = "NCT of Delhi (official rate card)"
d["drop_location"]   = np.nan
d["distance_km"]     = pd.to_numeric(d["distance_km"], errors="coerce")
d["fare_amount"]     = pd.to_numeric(d["fare_amount"], errors="coerce")
d["payment_method"]  = np.nan
d["status"]          = "official_rate_card"
d["driver_rating"]   = np.nan
d["customer_rating"] = np.nan
df_delhi_clean = make_common(d)

# --- 2f. Arunachal Pradesh route-fare table ---
d = df_aru.copy()
d["source_dataset"]  = "aru_dto_2019"
d["date"]            = pd.NaT
d["time"]            = np.nan
d["vehicle_type"]    = d["vehicle_type"]
d["pickup_location"] = d["from_stand"]
d["drop_location"]   = d["to_location"]
d["distance_km"]     = np.nan
d["fare_amount"]     = pd.to_numeric(d["fare_amount"], errors="coerce")
d["payment_method"]  = np.nan
d["status"]          = d["fare_type"]
d["driver_rating"]   = np.nan
d["customer_rating"] = np.nan
df_aru_clean = make_common(d)

# --- 2g. Indore Ola dataset ---
d = df_indore.copy()
d.columns = d.columns.str.strip()
d["source_dataset"]  = "indore_ola"
d["date"]            = pd.to_datetime(d["Date"], errors="coerce")
d["time"]            = d["Time"]
d["vehicle_type"]    = d["Vehicle Type"]
d["pickup_location"] = d["Pickup Location"]
d["drop_location"]   = d["Drop Location"]
d["distance_km"]     = pd.to_numeric(d["Ride Distance"], errors="coerce")
d["fare_amount"]     = pd.to_numeric(d["Booking Value"], errors="coerce")
d["payment_method"]  = np.nan
d["status"]          = d["Booking Status"]
d["driver_rating"]   = pd.to_numeric(d["Driver Ratings"], errors="coerce")
d["customer_rating"] = pd.to_numeric(d["Customer Rating"], errors="coerce")
df_indore_clean = make_common(d)

# ============================================================================
# 3. MERGE (CONCATENATE) ALL DATASETS INTO ONE
# ============================================================================
merged_df = pd.concat(
    [
        df_ola_clean,
        df_bookings_clean,
        df_ncr_clean,
        df_rides_clean,
        df_delhi_clean,
        df_aru_clean,
        df_indore_clean,
    ],
    ignore_index=True
)

print("Merged shape (before preprocessing):", merged_df.shape)
print(merged_df["source_dataset"].value_counts())

# ============================================================================
# 4. PRE-TRAINING PREPROCESSING
# ============================================================================

# 4a. Drop rows with no fare (target variable)
# merged_df = merged_df.dropna(subset=["fare_amount"])

# 4b. Remove impossible / junk values
# merged_df = merged_df[merged_df["fare_amount"] > 0]
# merged_df = merged_df[(merged_df["distance_km"].isna()) | (merged_df["distance_km"] >= 0)]

# 4c. Standardize text columns
for col in ["vehicle_type", "payment_method", "status"]:
    merged_df[col] = merged_df[col].astype(str).str.strip().str.lower()
    merged_df[col] = merged_df[col].replace({"nan": np.nan})

# 4d. Feature engineering from date/time
merged_df["date"] = pd.to_datetime(merged_df["date"], errors="coerce")
merged_df["year"]    = merged_df["date"].dt.year
merged_df["month"]   = merged_df["date"].dt.month
merged_df["day"]     = merged_df["date"].dt.day
merged_df["weekday"] = merged_df["date"].dt.day_name()

merged_df["time"] = pd.to_datetime(merged_df["time"], format="%H:%M:%S", errors="coerce").dt.hour
merged_df.rename(columns={"time": "hour"}, inplace=True)

# 4e. Handle duplicates
# merged_df = merged_df.drop_duplicates()

# 4f. Handle remaining missing values
merged_df["distance_km"]     = merged_df["distance_km"].fillna(merged_df["distance_km"].median())
merged_df["driver_rating"]   = merged_df["driver_rating"].fillna(merged_df["driver_rating"].median())
merged_df["customer_rating"] = merged_df["customer_rating"].fillna(merged_df["customer_rating"].median())
merged_df["payment_method"]  = merged_df["payment_method"].fillna("cash")
merged_df["status"]          = merged_df["status"].fillna("sucess")

# 4g. Outlier removal on fare_amount (IQR method)
# Q1 = merged_df["fare_amount"].quantile(0.25)
# Q3 = merged_df["fare_amount"].quantile(0.75)
# IQR = Q3 - Q1
# lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
# merged_df = merged_df[(merged_df["fare_amount"] >= lower) & (merged_df["fare_amount"] <= upper)]

print("Final shape after preprocessing:", merged_df.shape)
print(merged_df["source_dataset"].value_counts())
print(merged_df["vehicle_type"].value_counts())
print(merged_df.head())

# ============================================================================
# 5. SAVE
# ============================================================================
out_path = path + "merged_cab_fare_data_final.csv"
merged_df.to_csv(out_path, index=False)
print("Saved final merged dataset to:", out_path)

Loading raw files from: C:\Users\lenovo\OneDrive\Desktop\Folder\cab_bike_auto_fare_prediction\dataset
Merged shape (before preprocessing): (465073, 12)
source_dataset
ncr_events                 160609
bookings                   103024
indore_ola                 100000
rides_data                  50000
bengaluru_ola               49999
delhi_notification_2023       928
aru_dto_2019                  513
Name: count, dtype: int64
Final shape after preprocessing: (465073, 16)
source_dataset
ncr_events                 160609
bookings                   103024
indore_ola                 100000
rides_data                  50000
bengaluru_ola               49999
delhi_notification_2023       928
aru_dto_2019                  513
Name: count, dtype: int64
vehicle_type
auto             88548
bike             75247
ebike            47556
prime sedan      36477
prime plus       36322
prime suv        36145
mini             35642
go mini          31873
go sedan         29042
premier sedan    19453
c

In [10]:
merged_df

,source_dataset,date,hour,vehicle_type,pickup_location,drop_location,distance_km,fare_amount,payment_method,status,driver_rating,customer_rating,year,month,day,weekday
0,bengaluru_ola,2024-01-28,6.0,auto,Area-3,Area-2,28.500,868.06,wallet,success,4.4,4.4,2024.0,1.0,28.0,Sunday
1,bengaluru_ola,2024-01-26,3.0,mini,Area-7,Area-6,15.435,NaN,cash,cancelled by driver,4.2,4.2,2024.0,1.0,26.0,Friday
2,bengaluru_ola,2024-01-15,16.0,bike,Area-40,Area-24,15.435,NaN,cash,cancelled by driver,4.2,4.2,2024.0,1.0,15.0,Monday
3,bengaluru_ola,2024-01-02,22.0,prime sedan,Area-11,Area-24,15.435,NaN,cash,cancelled by driver,4.2,4.2,2024.0,1.0,2.0,Tuesday
4,bengaluru_ola,2024-01-30,22.0,bike,Area-41,Area-45,15.435,NaN,cash,incomplete,4.2,4.2,2024.0,1.0,30.0,Tuesday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465068,indore_ola,2025-01-02,NaN,prime sedan,Scheme No. 54,MR 10,17.730,908.00,cash,canceled by driver,4.2,4.2,2025.0,1.0,2.0,Thursday
465069,indore_ola,2025-01-30,NaN,bike,RNT Marg,Patel Nagar,18.340,676.00,cash,success,3.9,3.7,2025.0,1.0,30.0,Thursday
465070,indore_ola,2025-01-08,NaN,prime plus,Scheme No. 114,South Tukoganj,9.280,780.00,cash,canceled by driver,4.2,4.2,2025.0,1.0,8.0,Wednesday
465071,indore_ola,2025-01-12,NaN,mini,MR 4,Chhoti Gwaltoli,3.320,1852.00,cash,canceled by driver,4.2,4.2,2025.0,1.0,12.0,Sunday


In [11]:
merged_df.to_csv(
    r"C:\Users\lenovo\OneDrive\Desktop\Folder\cab_bike_auto_fare_prediction\dataset\merged_dataset.csv",
    index=False
)